In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [3]:
data_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
train_data = datasets.ImageFolder("DATASET/TRAIN", transform=data_transform)
test_data= datasets.ImageFolder("DATASET/TEST", transform=data_transform)

In [6]:
len(train_data)

22564

In [7]:
len(test_data)

2513

In [8]:
train_data[0][0].size()

torch.Size([3, 128, 128])

In [9]:
train_data_loader = DataLoader(train_data, shuffle=True, batch_size=128)
test_data_loader = DataLoader(test_data, shuffle=True, batch_size=128)

In [10]:
images, labels = next(iter(train_data_loader))

print("Batch shape:", images.shape)   # e.g. torch.Size([128, 3, 224, 224])
print("Labels shape:", labels.shape)  # e.g. torch.Size([128])
print("Labels:", labels)

Batch shape: torch.Size([128, 3, 128, 128])
Labels shape: torch.Size([128])
Labels: tensor([0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
        1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0,
        0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1,
        0, 1, 1, 0, 0, 1, 0, 1])


In [13]:
class ConvNeuralNetwork(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.linearForward = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )
    

    def forward(self, x):
        X = self.features(x)
        res = self.linearForward(X)
        return res

In [15]:
model = ConvNeuralNetwork().to(device)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [17]:
for epoch in range(10):

  total_loss = 0
  for img, label in train_data_loader:
    img, label = img.to(device), label.to(device)

    optimizer.zero_grad()

    output = model(img)
    loss = criterion(output, label)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  avg_loss = total_loss / len(train_data_loader)
  print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

Epoch: 1, Loss: 0.43221402218786337
Epoch: 2, Loss: 0.36804292287866947
Epoch: 3, Loss: 0.3378400145950964
Epoch: 4, Loss: 0.3239378651319924
Epoch: 5, Loss: 0.3093842675456893
Epoch: 6, Loss: 0.29655077438906763
Epoch: 7, Loss: 0.2919660859693915
Epoch: 8, Loss: 0.27672912146748796
Epoch: 9, Loss: 0.2662326800452787
Epoch: 10, Loss: 0.2523313695933186


In [18]:
model.eval()
images, labels = next(iter(test_data_loader))

with torch.no_grad():
    outputs = model(images.to(device))
    probs   = torch.softmax(outputs, dim=1)
    confs, preds = torch.max(probs, dim=1)
    print(confs, preds, labels)

tensor([0.9985, 0.6978, 0.9838, 0.9670, 0.6761, 0.9478, 0.9843, 0.9707, 0.9911,
        0.5139, 0.9714, 0.8696, 0.9289, 0.5019, 0.9946, 0.9970, 0.9955, 0.9862,
        0.8899, 0.7333, 0.9785, 0.9996, 0.9457, 0.9875, 0.9986, 0.8565, 0.9989,
        0.9994, 0.6773, 0.9548, 0.8203, 0.6874, 0.5895, 0.6570, 0.9999, 0.9780,
        0.9905, 0.9994, 0.8490, 0.6938, 0.9991, 0.9227, 0.5102, 0.7100, 0.9929,
        0.9281, 0.9999, 0.9946, 0.9207, 0.8604, 0.9223, 0.9980, 0.7843, 0.7417,
        0.9996, 0.7988, 0.9782, 0.9790, 0.8347, 0.6148, 0.9856, 0.9817, 0.8473,
        0.7196, 0.7987, 0.9971, 0.9898, 0.6237, 0.7051, 0.9975, 0.9945, 0.9662,
        0.5018, 0.9999, 0.9469, 0.9705, 0.9965, 0.6126, 0.9919, 0.9978, 0.9999,
        0.9604, 0.9636, 0.9903, 0.9990, 0.7374, 0.9941, 0.9935, 0.9565, 0.9933,
        0.9998, 0.9657, 0.9792, 0.8972, 0.9988, 0.9801, 0.9959, 0.9955, 0.9891,
        0.9993, 0.9842, 0.9254, 0.9957, 0.9972, 0.9934, 0.9069, 0.6881, 0.6772,
        0.7508, 0.9994, 0.9971, 0.9869, 